# EPIC Clarity Device Exposure Hydration

This notebook hydrates the OMOP DEVICE_EXPOSURE table from EPIC Clarity OR implant data.

## Source Table
- `_exponent._bronze_epic_clarity_*.dbo_OR_IMP`

In [ ]:
%sql
-- TRUNCATE Gold table for Epic (run this to clear stale data before reload)
TRUNCATE TABLE _exponent.omop_epic.device_exposure;

In [ ]:
%sql
-- Delete Epic records from Silver and Mapping tables (for full refresh)
DELETE FROM _exponent.omop_silver.device_exposure WHERE source_system = 'epic_clarity';

DELETE FROM _exponent.omop_mapping.source_to_device_exposure WHERE source_system = 'epic_clarity';

In [ ]:
%sql
-- Create silver_device_exposure temp view for Epic Clarity
CREATE OR REPLACE TEMPORARY VIEW silver_device_exposure AS
SELECT
  -- Person ID from source_to_person
  stp.person_id,
  -- Device concept (0 - no mapping available)
  0 AS device_concept_id,
  -- Start date (use received date or extract timestamp)
  DATE(COALESCE(oi.RECEIVED_DATE, oi.EXTRACT_FLAG_DTTM, oi.ETL_LOAD_TS)) AS device_exposure_start_date,
  COALESCE(oi.RECEIVED_DATE, oi.EXTRACT_FLAG_DTTM, oi.ETL_LOAD_TS) AS device_exposure_start_datetime,
  -- End date (use expiration or same as start)
  DATE(COALESCE(oi.EXPIRATION_DATE, oi.RECEIVED_DATE, oi.EXTRACT_FLAG_DTTM, oi.ETL_LOAD_TS)) AS device_exposure_end_date,
  COALESCE(oi.EXPIRATION_DATE, oi.RECEIVED_DATE, oi.EXTRACT_FLAG_DTTM, oi.ETL_LOAD_TS) AS device_exposure_end_datetime,
  -- Type
  32817 AS device_type_concept_id,  -- EHR
  -- Device details
  oi.STATIC_UDI AS unique_device_id,
  NULL AS production_id,
  TRY_CAST(oi.IMPLANT_VOLUME AS INT) AS quantity,
  NULL AS provider_id,
  NULL AS visit_occurrence_id,
  NULL AS visit_detail_id,
  -- Source values
  oi.IMPLANT_NAME AS device_source_value,
  0 AS device_source_concept_id,
  NULL AS unit_concept_id,
  NULL AS unit_source_value,
  NULL AS unit_source_concept_id,
  CONCAT_WS(CHR(31), 'epic_clarity', 'OR_IMP', 'IMPLANT_ID', CAST(oi.IMPLANT_ID AS STRING)) AS device_exposure_source_value,
  'epic_clarity' AS source_system
FROM _exponent._bronze_epic_clarity.or_imp oi
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', oi.PAT_ID)
  AND stp.active_flag = TRUE
WHERE oi.IMPLANT_ID IS NOT NULL
  AND oi.PAT_ID IS NOT NULL
  AND (oi.DELETE_FLAG = 0 OR oi.DELETE_FLAG IS NULL)

In [ ]:
%sql
-- Merge to Silver layer
MERGE INTO _exponent.omop_silver.device_exposure AS t
USING (
  SELECT * FROM (
    SELECT
      *,
      ROW_NUMBER() OVER (
        PARTITION BY device_exposure_source_value
        ORDER BY device_exposure_start_date DESC
      ) AS rn
    FROM silver_device_exposure
  ) WHERE rn = 1
) AS s
ON t.device_exposure_source_value = s.device_exposure_source_value

WHEN MATCHED AND (
     NOT (t.device_concept_id <=> s.device_concept_id)
  OR NOT (t.device_exposure_start_date <=> s.device_exposure_start_date)
  OR NOT (t.device_exposure_end_date <=> s.device_exposure_end_date)
  OR NOT (t.device_type_concept_id <=> s.device_type_concept_id)
  OR NOT (t.device_source_value <=> s.device_source_value)
  OR NOT (t.person_id <=> s.person_id)
)
THEN UPDATE SET
  t.person_id                   = s.person_id,
  t.device_concept_id           = s.device_concept_id,
  t.device_exposure_start_date  = s.device_exposure_start_date,
  t.device_exposure_start_datetime = s.device_exposure_start_datetime,
  t.device_exposure_end_date    = s.device_exposure_end_date,
  t.device_exposure_end_datetime = s.device_exposure_end_datetime,
  t.device_type_concept_id      = s.device_type_concept_id,
  t.unique_device_id            = s.unique_device_id,
  t.production_id               = s.production_id,
  t.quantity                    = s.quantity,
  t.provider_id                 = s.provider_id,
  t.visit_occurrence_id         = s.visit_occurrence_id,
  t.visit_detail_id             = s.visit_detail_id,
  t.device_source_value         = s.device_source_value,
  t.device_source_concept_id    = s.device_source_concept_id,
  t.unit_concept_id             = s.unit_concept_id,
  t.unit_source_value           = s.unit_source_value,
  t.unit_source_concept_id      = s.unit_source_concept_id,
  t.source_system               = s.source_system,
  t.last_mod_tsp                = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
  person_id,
  device_concept_id,
  device_exposure_start_date,
  device_exposure_start_datetime,
  device_exposure_end_date,
  device_exposure_end_datetime,
  device_type_concept_id,
  unique_device_id,
  production_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  device_source_value,
  device_source_concept_id,
  unit_concept_id,
  unit_source_value,
  unit_source_concept_id,
  device_exposure_source_value,
  source_system,
  last_mod_tsp
)
VALUES (
  s.person_id,
  s.device_concept_id,
  s.device_exposure_start_date,
  s.device_exposure_start_datetime,
  s.device_exposure_end_date,
  s.device_exposure_end_datetime,
  s.device_type_concept_id,
  s.unique_device_id,
  s.production_id,
  s.quantity,
  s.provider_id,
  s.visit_occurrence_id,
  s.visit_detail_id,
  s.device_source_value,
  s.device_source_concept_id,
  s.unit_concept_id,
  s.unit_source_value,
  s.unit_source_concept_id,
  s.device_exposure_source_value,
  s.source_system,
  current_timestamp()
);

In [ ]:
%sql
-- Insert new mappings to source_to_device_exposure
INSERT INTO _exponent.omop_mapping.source_to_device_exposure (
    source_system,
    device_exposure_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp
)
SELECT
    s.source_system,
    s.device_exposure_source_value,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    COALESCE(s.last_mod_tsp, current_timestamp()) AS last_mod_tsp
FROM (
    SELECT DISTINCT source_system, device_exposure_source_value, last_mod_tsp
    FROM _exponent.omop_silver.device_exposure
    WHERE source_system = 'epic_clarity'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_device_exposure x
  ON s.device_exposure_source_value = x.device_exposure_source_value;

In [ ]:
%sql
-- Merge to Gold layer
MERGE INTO _exponent.omop_epic.device_exposure AS gold
USING (
  SELECT
    sde.device_exposure_id,
    s.person_id,
    s.device_concept_id,
    s.device_exposure_start_date,
    s.device_exposure_start_datetime,
    s.device_exposure_end_date,
    s.device_exposure_end_datetime,
    s.device_type_concept_id,
    s.unique_device_id,
    s.production_id,
    s.quantity,
    s.provider_id,
    s.visit_occurrence_id,
    s.visit_detail_id,
    s.device_source_value,
    s.device_source_concept_id,
    s.unit_concept_id,
    s.unit_source_value,
    s.unit_source_concept_id
  FROM _exponent.omop_silver.device_exposure s
  JOIN _exponent.omop_mapping.source_to_device_exposure sde
    ON sde.device_exposure_source_value = s.device_exposure_source_value
   AND sde.active_flag = TRUE
  WHERE s.source_system = 'epic_clarity'
) AS src
ON gold.device_exposure_id = src.device_exposure_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id                    = src.person_id,
  gold.device_concept_id            = src.device_concept_id,
  gold.device_exposure_start_date   = src.device_exposure_start_date,
  gold.device_exposure_start_datetime = src.device_exposure_start_datetime,
  gold.device_exposure_end_date     = src.device_exposure_end_date,
  gold.device_exposure_end_datetime = src.device_exposure_end_datetime,
  gold.device_type_concept_id       = src.device_type_concept_id,
  gold.unique_device_id             = src.unique_device_id,
  gold.production_id                = src.production_id,
  gold.quantity                     = src.quantity,
  gold.provider_id                  = src.provider_id,
  gold.visit_occurrence_id          = src.visit_occurrence_id,
  gold.visit_detail_id              = src.visit_detail_id,
  gold.device_source_value          = src.device_source_value,
  gold.device_source_concept_id     = src.device_source_concept_id,
  gold.unit_concept_id              = src.unit_concept_id,
  gold.unit_source_value            = src.unit_source_value,
  gold.unit_source_concept_id       = src.unit_source_concept_id

WHEN NOT MATCHED THEN INSERT (
  device_exposure_id,
  person_id,
  device_concept_id,
  device_exposure_start_date,
  device_exposure_start_datetime,
  device_exposure_end_date,
  device_exposure_end_datetime,
  device_type_concept_id,
  unique_device_id,
  production_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  device_source_value,
  device_source_concept_id,
  unit_concept_id,
  unit_source_value,
  unit_source_concept_id
)
VALUES (
  src.device_exposure_id,
  src.person_id,
  src.device_concept_id,
  src.device_exposure_start_date,
  src.device_exposure_start_datetime,
  src.device_exposure_end_date,
  src.device_exposure_end_datetime,
  src.device_type_concept_id,
  src.unique_device_id,
  src.production_id,
  src.quantity,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.device_source_value,
  src.device_source_concept_id,
  src.unit_concept_id,
  src.unit_source_value,
  src.unit_source_concept_id
);

In [ ]:
%sql
-- Validation queries
-- 1. Check person_id FK integrity (should be 0 orphan records)
SELECT 'DEVICE_EXPOSURE.PERSON_ID FK' as check_field,
       COUNT(*) as orphan_records
FROM _exponent.omop_epic.device_exposure de
WHERE NOT EXISTS (SELECT 1 FROM _exponent.omop_epic.person p WHERE p.person_id = de.person_id);

-- 2. Check type_concept_id validity
SELECT 'DEVICE_EXPOSURE.DEVICE_TYPE_CONCEPT_ID' as check_field,
       COUNT(*) as total,
       SUM(CASE WHEN device_type_concept_id = 0 THEN 1 ELSE 0 END) as zero_type
FROM _exponent.omop_epic.device_exposure;

-- 3. Total record count
SELECT 'total_records' as check_field, COUNT(*) as cnt FROM _exponent.omop_epic.device_exposure;